# 02 · Thiết kế Semantic Model

Notebook tạo prompt AI 2 từ `requirement.json` và `data_catalog.json`, sau đó kiểm tra cấu trúc `model_spec.json`. Thiết kế model phải dùng đúng các bảng/cột đã quét; nội dung DAX vẫn cần được người phụ trách nghiệp vụ duyệt.

In [ ]:
from pathlib import Path
import json

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy repo root.')

ROOT = find_repo_root()
RUNS_DIR = ROOT / 'runs'
JOB_ID = None  # Có thể đặt chính xác job_id; None sẽ chọn job mới nhất có requirement.json.
if JOB_ID:
    RUN_DIR = RUNS_DIR / JOB_ID
else:
    candidates = [p for p in RUNS_DIR.glob('*') if (p / 'requirement.json').exists()]
    if not candidates:
        raise FileNotFoundError('Chưa có requirement.json. Chạy 01_intake_scan.ipynb trước.')
    RUN_DIR = max(candidates, key=lambda p: p.stat().st_mtime)
JOB_ID = RUN_DIR.name
requirement = json.loads((RUN_DIR / 'requirement.json').read_text(encoding='utf-8'))
catalog = json.loads((RUN_DIR / 'data_catalog.json').read_text(encoding='utf-8'))
source_tables = {table['name']: {column['name'] for column in table['columns']} for table in catalog['tables']}
print('Job:', JOB_ID, '| Bảng nguồn:', ', '.join(source_tables))

## AI 2 · Đề xuất model
Yêu cầu AI trả về JSON duy nhất theo schema trong prompt. Với mỗi cột nguồn phải giữ `source_table` và `source_column`; cột tính toán phải có `expression`. Mỗi measure cần DAX và bảng chứa measure.

In [ ]:
prompt = f'''Thiết kế Power BI semantic model cho yêu cầu sau. Chỉ sử dụng cột có trong data catalog; không tạo quan hệ nếu không có khóa phù hợp.
Trả về JSON duy nhất: {{"model_name": "...", "tables": [{{"name": "...", "source_table": "tên bảng catalog", "columns": [{{"name": "...", "source_column": "cột catalog hoặc null", "data_type": "string|int64|double|dateTime|boolean", "role": "key|dimension|measure", "expression": "biểu thức tùy chọn"}}]}}], "relationships": [{{"from_table": "...", "from_column": "...", "to_table": "...", "to_column": "...", "cardinality": "manyToOne", "is_active": true}}], "measures": [{{"name": "...", "table": "...", "expression": "DAX", "format_string": "..."}}], "assumptions": ["..."]}}.
Không trả lời markdown/code fence. Ghi chú mọi giả định và tránh DAX không thể suy ra từ yêu cầu.

REQUIREMENT:
{json.dumps(requirement, ensure_ascii=False, indent=2)}

DATA CATALOG:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
'''
print(prompt)

In [ ]:
AI2_RESPONSE = r'''PASTE_AI_JSON_HERE'''

def validate_model(spec):
    if not isinstance(spec.get('tables'), list) or not spec['tables']:
        raise ValueError('model_spec phải có ít nhất một table.')
    model_tables = {table['name']: table for table in spec['tables']}
    if len(model_tables) != len(spec['tables']):
        raise ValueError('Tên table bị trùng.')
    for table in spec['tables']:
        source_name = table.get('source_table')
        if source_name not in source_tables:
            raise ValueError(f'Table {table["name"]}: source_table không có trong catalog: {source_name}')
        known_columns = source_tables[source_name]
        names = set()
        for column in table.get('columns', []):
            if column['name'] in names:
                raise ValueError(f'Tên cột bị trùng trong {table["name"]}: {column["name"]}')
            names.add(column['name'])
            source_column = column.get('source_column')
            expression = column.get('expression')
            if source_column and source_column not in known_columns:
                raise ValueError(f'Cột nguồn không tồn tại: {source_name}[{source_column}]')
            if not source_column and not expression:
                raise ValueError(f'Cột {column["name"]} cần source_column hoặc expression.')
    for relationship in spec.get('relationships', []):
        if relationship.get('from_table') not in model_tables or relationship.get('to_table') not in model_tables:
            raise ValueError(f'Relationship tham chiếu table không tồn tại: {relationship}')
    for measure in spec.get('measures', []):
        if measure.get('table') not in model_tables or not measure.get('expression', '').strip():
            raise ValueError(f'Measure thiếu table hoặc DAX expression: {measure}')

if AI2_RESPONSE.strip() != 'PASTE_AI_JSON_HERE':
    model_spec = json.loads(AI2_RESPONSE)
    validate_model(model_spec)
    model_spec['source_job_id'] = JOB_ID
    (RUN_DIR / 'model_spec.json').write_text(json.dumps(model_spec, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã kiểm tra và ghi:', RUN_DIR / 'model_spec.json')
else:
    print('Dán JSON AI 2 vào AI2_RESPONSE để kiểm tra và lưu model_spec.json.')